In [ ]:
import altair as alt
import polars as pl
import polars.selectors as cs

In [ ]:
filepath = r"\path\to\bayview-geotab-by_time_period.csv"
time_period_order = ["ON", "EA", "AM", "MD", "PM", "EV"]
df = pl.read_csv(filepath)

In [ ]:
# add columns for vocations and NAICS/industries
df = (
    df.with_columns(
        pl.col("ObservedCountByVocation")
        .str.replace_all("'", '"')
        .str.replace_all("None", "null")
        .str.json_decode(
            pl.List(
                pl.Struct(
                    [
                        pl.Field("VocationId", pl.Int64),
                        pl.Field("Vocation", pl.String),
                        pl.Field("ObservedCount", pl.Int64),
                        pl.Field("VehicleCount", pl.Int64),
                    ]
                )
            )
        )
        .list.eval(
            pl.struct(
                vocation=pl.concat_str(
                    [
                        pl.lit("vocation_"),
                        pl.element().struct.field("VocationId"),
                        pl.lit("_"),
                        pl.element().struct.field("Vocation"),
                    ]
                ),
                vocation_count=pl.element().struct.field("ObservedCount"),
            )
        ),
        pl.col("ObservedCountByIndustry")
        .str.replace_all("'", '"')
        .str.replace_all("None", "null")
        .str.json_decode(
            pl.List(
                pl.Struct(
                    [
                        pl.Field("NAICS_Code_1", pl.Int64),
                        pl.Field("ObservedCount", pl.Int64),
                        pl.Field("VehicleCount", pl.Int64),
                    ]
                )
            )
        )
        .list.eval(
            pl.struct(
                naics_code=pl.concat_str(
                    [
                        pl.lit("naics_"),
                        pl.element().struct.field("NAICS_Code_1"),
                    ]
                ),
                naics_count=pl.element().struct.field("ObservedCount"),
            )
        ),
    )
    .explode("ObservedCountByVocation")
    .unnest("ObservedCountByVocation")
    .pivot(
        index=cs.exclude("vocation", "vocation_count"),
        on="vocation",
        values="vocation_count",
        aggregate_function="first",
    )
    .drop("null")
    .explode("ObservedCountByIndustry")
    .unnest("ObservedCountByIndustry")
    .pivot(
        index=cs.exclude("naics_code", "naics_count"),
        on="naics_code",
        values="naics_count",
        aggregate_function="first",
    )
    .drop("null")
)

In [ ]:
vocation_df = (
    df.group_by("Time Period")
    .agg(pl.sum("ObservedCount"), cs.starts_with("vocation").sum())
    .with_columns(total_vocation_counts=pl.sum_horizontal(cs.starts_with("vocation")))
)
vocation_df
# TODO !!! where are all these missing vocation counts:
# total_vocation_counts vs ObservedCount? probably from the nulls

In [ ]:
vocation_shares_df = vocation_df.with_columns(
    cs.starts_with("vocation") / pl.col("total_vocation_counts")
).unpivot(
    index="Time Period",
    on=cs.starts_with("vocation"),
    variable_name="vocation",
    value_name="vocation_share",
)
# vocation_shares_df

In [ ]:
vocation_chart = (
    alt.Chart(vocation_shares_df)
    .mark_bar()
    .encode(
        x=alt.X("Time Period:N", sort=time_period_order),
        y=alt.Y("vocation_share:Q"),
        color=alt.Color("vocation:N"),
        tooltip=["Time Period", "vocation", "vocation_share"],
    )
)
vocation_chart

In [ ]:
industry_df = (
    df.group_by("Time Period")
    .agg(pl.sum("ObservedCount"), cs.starts_with("naics").sum())
    .with_columns(total_naics_counts=pl.sum_horizontal(cs.starts_with("naics")))
)
industry_df
# TODO !!! where are all these missing industry counts:
# total_naics_counts vs ObservedCount? probably from the nulls

In [ ]:
industry_shares_df = industry_df.with_columns(
    cs.starts_with("naics") / pl.col("total_naics_counts")
).unpivot(
    index="Time Period",
    on=cs.starts_with("naics"),
    variable_name="industry",
    value_name="naics_share",
)

In [ ]:
industry_chart = (
    alt.Chart(industry_shares_df)
    .mark_bar()
    .encode(
        x=alt.X("Time Period:N", sort=time_period_order),
        y=alt.Y("naics_share:Q", scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("industry:N"),
        tooltip=["Time Period", "industry", "naics_share"],
    )
)
industry_chart